In [1]:
from pathlib import Path

import pandas as pd


def find_project_root(start_path):
    """Find the project folder containing the .git directory."""
    start_path = Path(start_path).resolve()

    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder

    raise FileNotFoundError("Could not find the project root folder.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)

Project root: /Users/imseoyeong/Desktop/east-coast-job-market-analysis
Raw data folder: /Users/imseoyeong/Desktop/east-coast-job-market-analysis/data/raw


In [2]:
raw_files = sorted(RAW_DATA_DIR.glob("adzuna_jobs_*.csv"))

if not raw_files:
    raise FileNotFoundError("No Adzuna CSV files were found.")

latest_file = raw_files[-1]

jobs = pd.read_csv(latest_file)

print("Loaded file:", latest_file.name)
print("Dataset shape:", jobs.shape)

Loaded file: adzuna_jobs_20260804_2023.csv
Dataset shape: (492, 19)


In [3]:
jobs.head()

,job_id,title,company,location,location_area,query_state,query_location,search_term,category,description,created,salary_min,salary_max,contract_time,contract_type,latitude,longitude,job_url,collected_at_utc
0,5822586053,Data Analyst,RightTalents,"Flatbush, Brooklyn",US | New York | New York City | Brooklyn | Fla...,NY,New York,data analyst,IT Jobs,Role: Data Analyst Location: Brooklyn NY (Onsi...,2026-07-31T13:19:47Z,132558.17,132558.17,NaN,NaN,40.627791,-73.946201,https://www.adzuna.com/land/ad/5822586053?se=c...,2026-08-05T01:23:03.854493+00:00
1,5823519927,Data Analyst,EXL Service,"New York City, New York",US | New York | New York City,NY,New York,data analyst,Consultancy Jobs,EXL (NASDAQ:EXLS) is a leading operations mana...,2026-08-01T03:55:02Z,250.00,250.00,NaN,NaN,NaN,NaN,https://www.adzuna.com/land/ad/5823519927?se=c...,2026-08-05T01:23:03.854553+00:00
2,5806988210,Associate Director - Data Analyst,Moody's Corporation,"New York City, New York",US | New York | New York City,NY,New York,data analyst,IT Jobs,hackajob is collaborating with Moody's Corpora...,2026-07-19T04:52:16Z,171285.89,171285.89,NaN,NaN,40.767036,-73.970368,https://www.adzuna.com/land/ad/5806988210?se=c...,2026-08-05T01:23:03.854571+00:00
3,5820553649,Senior Data Analyst,Spruce Technology Inc.,"Grand Central, Manhattan",US | New York | New York City | Manhattan | Gr...,NY,New York,data analyst,IT Jobs,Title: Sr. Data Analyst Location: Brooklyn NY ...,2026-07-29T21:31:13Z,157959.61,157959.61,NaN,NaN,40.752840,-73.975280,https://www.adzuna.com/land/ad/5820553649?se=c...,2026-08-05T01:23:03.854587+00:00
4,5820119810,Senior Data Analyst,Innovee Consulting LLC,"Flatbush, Brooklyn",US | New York | New York City | Brooklyn | Fla...,NY,New York,data analyst,IT Jobs,"Role: Senior Data Analyst Location: Brooklyn, ...",2026-07-29T13:42:48Z,120789.21,120789.21,NaN,NaN,40.627791,-73.946201,https://www.adzuna.com/land/ad/5820119810?se=c...,2026-08-05T01:23:03.854601+00:00


In [4]:
print("Columns:")
for column in jobs.columns:
    print("-", column)

print()
print("Number of rows:", len(jobs))
print("Number of columns:", len(jobs.columns))
print("Unique job IDs:", jobs["job_id"].nunique())
print("Duplicate job IDs:", jobs["job_id"].duplicated().sum())

Columns:
- job_id
- title
- company
- location
- location_area
- query_state
- query_location
- search_term
- category
- description
- created
- salary_min
- salary_max
- contract_time
- contract_type
- latitude
- longitude
- job_url
- collected_at_utc

Number of rows: 492
Number of columns: 19
Unique job IDs: 437
Duplicate job IDs: 55


In [5]:
missing_values = (
    jobs.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_values["missing_percent"] = (
    missing_values["missing_count"] / len(jobs) * 100
).round(1)

missing_values

,missing_count,missing_percent
contract_type,468,95.1
contract_time,362,73.6
longitude,38,7.7
latitude,38,7.7
job_id,0,0.0
created,0,0.0
job_url,0,0.0
salary_max,0,0.0
salary_min,0,0.0
description,0,0.0


In [6]:
duplicate_counts = jobs["job_id"].value_counts()

print("Number of unique job IDs:", jobs["job_id"].nunique())
print("Number of duplicated rows:", jobs["job_id"].duplicated().sum())
print()

duplicate_counts.value_counts().sort_index()

Number of unique job IDs: 437
Number of duplicated rows: 55



count
1    382
2     55
Name: count, dtype: int64

In [7]:
def join_unique(values):
    """Combine unique non-missing values into one string."""
    cleaned_values = values.dropna().astype(str)
    return " | ".join(sorted(set(cleaned_values)))


match_information = (
    jobs.groupby("job_id")
    .agg(
        matched_search_terms=("search_term", join_unique),
        matched_query_states=("query_state", join_unique),
        matched_query_locations=("query_location", join_unique),
        times_returned=("job_id", "size"),
    )
    .reset_index()
)

match_information.head()

,job_id,matched_search_terms,matched_query_states,matched_query_locations,times_returned
0,4687974701,data analyst | marketing analyst,NY,New York,2
1,4804431378,reporting analyst,NY,New York,1
2,4809682356,marketing analyst,NJ,New Jersey,1
3,4809780912,business intelligence analyst,DC | VA,"Virginia | Washington, DC",2
4,4809969784,business intelligence analyst,NY,New York,1


In [8]:
jobs_unique = (
    jobs.sort_values("created", ascending=False)
    .drop_duplicates(subset="job_id", keep="first")
    .drop(
        columns=[
            "search_term",
            "query_state",
            "query_location",
        ]
    )
    .merge(
        match_information,
        on="job_id",
        how="left",
    )
    .reset_index(drop=True)
)

print("Rows before removing duplicates:", len(jobs))
print("Rows after removing duplicates:", len(jobs_unique))
print("Duplicate job IDs remaining:", jobs_unique["job_id"].duplicated().sum())

Rows before removing duplicates: 492
Rows after removing duplicates: 437
Duplicate job IDs remaining: 0


In [9]:
jobs_unique[
    [
        "title",
        "company",
        "location",
        "matched_search_terms",
        "matched_query_states",
        "times_returned",
    ]
].head(10)

,title,company,location,matched_search_terms,matched_query_states,times_returned
0,Data Analyst,Brooksource,"New Jersey, US",data analyst,NJ,1
1,Healthcare Reporting Analyst,"Charter Global, Inc.","East Case, Baltimore",reporting analyst,MD,1
2,Reporting Analyst,System One,"Druid, Baltimore",reporting analyst,MD,1
3,Big Data Analyst,Vantor,"Herndon, Fairfax County",data analyst,VA,1
4,Senior Data Analyst with DataStage,InfoPeople Corp,"Flatbush, Brooklyn",data analyst,NY,1
5,Reporting Analyst,Vega Consulting Solutions,"East Case, Baltimore",reporting analyst,MD,1
6,Program Operations Analyst with Security Clear...,Quantum Sky,"Suitland, Prince George's County",operations analyst,DC,1
7,Data Analyst,International Solutions Group,"Pimmit, Fairfax County",data analyst,DC | VA,2
8,Mainframe Data Analyst,TECH Tammina,"Flatbush, Brooklyn",data analyst,NY,1
9,Reporting Analyst,Lumen Solutions Group Inc.,"East Case, Baltimore",reporting analyst,MD,1


In [10]:
jobs_unique.loc[
    jobs_unique["times_returned"] > 1,
    [
        "title",
        "company",
        "matched_search_terms",
        "matched_query_states",
        "times_returned",
    ],
].sort_values(
    "times_returned",
    ascending=False,
).head(20)

,title,company,matched_search_terms,matched_query_states,times_returned
7,Data Analyst,International Solutions Group,data analyst,DC | VA,2
404,Marketing Operations Analyst,FocusKPI,marketing analyst | operations analyst,MD,2
296,"BI Reporting Analyst, TPO Data & Insights",Marriott,reporting analyst,DC | MD,2
307,"Senior Analyst, Marketing Investment & Planning",GEICO,marketing analyst,DC | MD,2
309,Data Analyst,Cherokee Federal,data analyst,DC | VA,2
310,"Analyst, Digital Marketing",Stagwell Global LLC,marketing analyst,DC | VA,2
336,Mid-Level Fullstack Software Engineer,Software Guidance & Assistance,business intelligence analyst,NJ | NY,2
353,Sr. Sales Operations Analyst,Danaher Corporation,operations analyst,NJ | NY,2
377,Data Analyst,Chenega MIOS,data analyst,DC | VA,2
393,Senior Business Intelligence Analyst,Technomics,business intelligence analyst,DC | VA,2


In [12]:
import html
import re


def normalize_text(value):
    """Normalize text for duplicate comparison."""
    if pd.isna(value):
        return ""

    text = html.unescape(str(value)).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


jobs_unique["title_norm"] = jobs_unique["title"].apply(normalize_text)
jobs_unique["company_norm"] = jobs_unique["company"].apply(normalize_text)
jobs_unique["location_norm"] = jobs_unique["location"].apply(normalize_text)
jobs_unique["description_norm"] = jobs_unique["description"].apply(
    normalize_text
)

jobs_unique["salary_min_key"] = (
    jobs_unique["salary_min"]
    .fillna(-1)
    .round(2)
    .astype(str)
)

jobs_unique["salary_max_key"] = (
    jobs_unique["salary_max"]
    .fillna(-1)
    .round(2)
    .astype(str)
)

jobs_unique["created_key"] = (
    pd.to_datetime(
        jobs_unique["created"],
        errors="coerce",
        utc=True,
    )
    .astype(str)
)

duplicate_key_columns = [
    "title_norm",
    "company_norm",
    "location_norm",
    "description_norm",
    "salary_min_key",
    "salary_max_key",
    "created_key",
]

jobs_unique["strict_duplicate_key"] = (
    jobs_unique[duplicate_key_columns]
    .astype(str)
    .agg(" || ".join, axis=1)
)

In [13]:
strict_duplicates = jobs_unique.loc[
    jobs_unique.duplicated(
        subset="strict_duplicate_key",
        keep=False,
    )
].sort_values(
    [
        "company",
        "title",
        "location",
    ]
)

print("Rows involved in strict duplicates:", len(strict_duplicates))
print(
    "Strict duplicate groups:",
    strict_duplicates["strict_duplicate_key"].nunique(),
)

strict_duplicates[
    [
        "job_id",
        "title",
        "company",
        "location",
        "matched_query_states",
        "created",
        "salary_min",
        "salary_max",
    ]
].head(30)

Rows involved in strict duplicates: 0
Strict duplicate groups: 0


,job_id,title,company,location,matched_query_states,created,salary_min,salary_max


In [14]:
def combine_pipe_values(series):
    """Combine values separated by | without repetition."""
    combined = set()

    for value in series.dropna().astype(str):
        for item in value.split("|"):
            item = item.strip()

            if item:
                combined.add(item)

    return " | ".join(sorted(combined))


duplicate_metadata = (
    jobs_unique.groupby("strict_duplicate_key")
    .agg(
        all_job_ids=(
            "job_id",
            lambda values: " | ".join(
                sorted(set(values.astype(str)))
            ),
        ),
        merged_search_terms=(
            "matched_search_terms",
            combine_pipe_values,
        ),
        merged_query_states=(
            "matched_query_states",
            combine_pipe_values,
        ),
        merged_query_locations=(
            "matched_query_locations",
            combine_pipe_values,
        ),
        total_times_returned=(
            "times_returned",
            "sum",
        ),
        duplicate_record_count=(
            "job_id",
            "size",
        ),
    )
    .reset_index()
)

jobs_clean = (
    jobs_unique.sort_values(
        "created",
        ascending=False,
    )
    .drop_duplicates(
        subset="strict_duplicate_key",
        keep="first",
    )
    .drop(
        columns=[
            "matched_search_terms",
            "matched_query_states",
            "matched_query_locations",
            "times_returned",
        ]
    )
    .merge(
        duplicate_metadata,
        on="strict_duplicate_key",
        how="left",
    )
    .rename(
        columns={
            "merged_search_terms": "matched_search_terms",
            "merged_query_states": "matched_query_states",
            "merged_query_locations": "matched_query_locations",
            "total_times_returned": "times_returned",
        }
    )
    .reset_index(drop=True)
)

In [15]:
print("Rows after job_id deduplication:", len(jobs_unique))
print("Rows after strict deduplication:", len(jobs_clean))
print(
    "Additional duplicate rows removed:",
    len(jobs_unique) - len(jobs_clean),
)
print(
    "Duplicate job IDs remaining:",
    jobs_clean["job_id"].duplicated().sum(),
)
print(
    "Merged duplicate groups:",
    (jobs_clean["duplicate_record_count"] > 1).sum(),
)

Rows after job_id deduplication: 437
Rows after strict deduplication: 437
Additional duplicate rows removed: 0
Duplicate job IDs remaining: 0
Merged duplicate groups: 0


In [22]:
def classify_title_seniority(title):
    """Classify seniority using keywords in the job title."""
    title_text = normalize_text(title)

    executive_pattern = (
        r"\b(director|vice president|vp|chief|head)\b"
    )
    manager_pattern = (
        r"\b(manager|supervisor)\b"
    )
    senior_pattern = (
        r"\b(senior|sr\.?|lead|principal)\b"
    )
    internship_pattern = (
        r"\b(intern|internship|co-op)\b"
    )
    entry_pattern = (
        r"\b(junior|jr\.?|entry[- ]level|"
        r"associate|trainee|graduate)\b"
    )
    staff_pattern = r"\bstaff\b"

    if re.search(executive_pattern, title_text):
        return "Director or Executive"

    if re.search(manager_pattern, title_text):
        return "Manager"

    if re.search(senior_pattern, title_text):
        return "Senior or Lead"

    if re.search(internship_pattern, title_text):
        return "Internship"

    if re.search(entry_pattern, title_text):
        return "Junior or Entry"

    if re.search(staff_pattern, title_text):
        return "Staff or Unclear"

    return "Unspecified"


jobs_clean["title_seniority"] = jobs_clean["title"].apply(
    classify_title_seniority
)

In [23]:
seniority_counts = (
    jobs_clean["title_seniority"]
    .value_counts()
    .rename_axis("title_seniority")
    .reset_index(name="job_count")
)

seniority_counts["percent"] = (
    seniority_counts["job_count"] / len(jobs_clean) * 100
).round(1)

seniority_counts

,title_seniority,job_count,percent
0,Unspecified,315,72.1
1,Senior or Lead,87,19.9
2,Manager,10,2.3
3,Director or Executive,10,2.3
4,Internship,8,1.8
5,Junior or Entry,7,1.6


In [24]:
for seniority_level in jobs_clean["title_seniority"].unique():
    print("=" * 60)
    print(seniority_level)
    print("=" * 60)

    examples = (
        jobs_clean.loc[
            jobs_clean["title_seniority"] == seniority_level,
            "title",
        ]
        .drop_duplicates()
        .head(15)
    )

    for title in examples:
        print("-", title)

    print()

Unspecified
- Data Analyst
- Healthcare Reporting Analyst
- Reporting Analyst
- Big Data Analyst
- Program Operations Analyst with Security Clearance
- Mainframe Data Analyst
- Business Intelligence Analyst
- Data Analyst with Data Modelling and Pension/Retirement
- Radiologic Technologist
- Healthcare Reporting Data Analyst
- IT Data Analyst
- Remote Investor Reporting Analyst
- Operations Analyst
- Business Intelligence Developer/Business Analyst - PFS
- Sales Business Intelligence Analyst

Senior or Lead
- Senior Data Analyst with DataStage
- Senior Equity Analyst & Reporter - CNBC
- Submarine Logistics Data Analyst, Lead
- Senior Data Analyst with ETL expertise
- Senior Business Intelligence Analyst - Campaign Delivery
- Senior Business Intelligence & Transaction Analyst
- Investment Research Analyst/Senior Investment Research Analyst (Contractual) - FIN
- Sr. Business Data Analyst
- PIM - Sr Databricks & Reporting Analyst
- Senior Data Analyst
- Senior Business Intelligence Data A

In [19]:
advanced_levels = [
    "Director or Executive",
    "Manager",
    "Senior or Lead",
]

jobs_clean["advanced_title_flag"] = (
    jobs_clean["title_seniority"].isin(advanced_levels)
)

print(
    "Clearly advanced-level titles:",
    jobs_clean["advanced_title_flag"].sum(),
)

print(
    "Remaining titles:",
    (~jobs_clean["advanced_title_flag"]).sum(),
)

Clearly advanced-level titles: 107
Remaining titles: 330


In [25]:
jobs_clean.loc[
    jobs_clean["advanced_title_flag"],
    [
        "title",
        "company",
        "location",
        "title_seniority",
    ],
].sort_values(
    ["title_seniority", "title"]
).head(30)

,title,company,location,title_seniority
148,Associate Director - Data Analyst,Moody's Corporation,"New York City, New York",Director or Executive
177,Associate Director of Digital Marketing,Worldwide Assurance for Employees of Public Ag...,"West Falls Church, Fairfax County",Director or Executive
80,"Associate Director, Strategic Options and Asse...",Bristol Myers Squibb,"Princeton, Mercer County",Director or Executive
102,"Associate Director, Strategic Options and Asse...",Bristol Myers Squibb,"Princeton, Mercer County",Director or Executive
125,Chief Enterprise Architect (Business Intellige...,BTI,"Rosslyn, Arlington County",Director or Executive
135,Chief Enterprise Architect (Business Intellige...,BTI,"Washington, Washington, D.C.",Director or Executive
113,Chief Enterprise Business Architect (Business ...,BTI,"Rosslyn, Arlington County",Director or Executive
129,Chief Enterprise Business Architect (Business ...,BTI,"Washington, Washington, D.C.",Director or Executive
388,"VP, Liquidity Reporting Analyst",Crédit Agricole CIB,"New York City, New York",Director or Executive
391,"VP, Regulatory Reporting Analyst",Crédit Agricole CIB,"New York City, New York",Director or Executive


In [27]:
advanced_levels = [
    "Director or Executive",
    "Manager",
    "Senior or Lead",
]

In [28]:
def extract_experience_years(description):
    """
    Extract the lowest stated years of experience from a job description.

    This is a heuristic, so results should be treated as estimates.
    """
    text = normalize_text(description)

    patterns = [
        # Examples: 3-5 years of experience, 2 to 4 years experience
        r"\b(\d{1,2})\s*(?:-|–|—|to)\s*\d{1,2}\s*"
        r"(?:years?|yrs?)\s+(?:of\s+)?"
        r"(?:relevant\s+|related\s+|professional\s+|work\s+)?"
        r"experience\b",

        # Examples: 3+ years of experience, 5 years experience
        r"\b(\d{1,2})\s*(?:\+|plus)?\s*"
        r"(?:years?|yrs?)\s+(?:of\s+)?"
        r"(?:relevant\s+|related\s+|professional\s+|work\s+)?"
        r"experience\b",

        # Examples: minimum of 3 years, at least 2 years
        r"\b(?:minimum(?: of)?|at least)\s+(\d{1,2})\s*"
        r"(?:years?|yrs?)\b",
    ]

    years_found = []

    for pattern in patterns:
        matches = re.findall(pattern, text)

        for match in matches:
            years_found.append(int(match))

    if not years_found:
        return pd.NA

    return min(years_found)


jobs_clean["minimum_experience_years"] = (
    jobs_clean["description"]
    .apply(extract_experience_years)
    .astype("Int64")
)

In [29]:
print(
    "Descriptions with extracted experience:",
    jobs_clean["minimum_experience_years"].notna().sum(),
)

print(
    "Descriptions without extracted experience:",
    jobs_clean["minimum_experience_years"].isna().sum(),
)

jobs_clean["minimum_experience_years"].value_counts(
    dropna=False
).sort_index()

Descriptions with extracted experience: 9
Descriptions without extracted experience: 428


minimum_experience_years
0         1
1         1
2         2
3         1
8         1
10        1
25        2
<NA>    428
Name: count, dtype: Int64

In [30]:
def classify_experience_level(row):
    title_level = row["title_seniority"]
    years = row["minimum_experience_years"]

    if title_level == "Internship":
        return "Internship"

    if title_level == "Junior or Entry":
        return "Likely Entry-Level"

    if title_level in [
        "Director or Executive",
        "Manager",
        "Senior or Lead",
    ]:
        return "Advanced-Level"

    if pd.isna(years):
        return "Unclear"

    if years <= 2:
        return "Likely Entry-Level"

    if years <= 4:
        return "Mid-Level"

    return "Advanced-Level"


jobs_clean["experience_level"] = jobs_clean.apply(
    classify_experience_level,
    axis=1,
)

In [31]:
experience_counts = (
    jobs_clean["experience_level"]
    .value_counts()
    .rename_axis("experience_level")
    .reset_index(name="job_count")
)

experience_counts["percent"] = (
    experience_counts["job_count"]
    / len(jobs_clean)
    * 100
).round(1)

experience_counts

,experience_level,job_count,percent
0,Unclear,308,70.5
1,Advanced-Level,109,24.9
2,Likely Entry-Level,11,2.5
3,Internship,8,1.8
4,Mid-Level,1,0.2


In [32]:
entry_level_jobs = jobs_clean.loc[
    jobs_clean["experience_level"] == "Likely Entry-Level"
].copy()

print("Likely entry-level postings:", len(entry_level_jobs))

entry_level_jobs[
    [
        "title",
        "company",
        "location",
        "title_seniority",
        "minimum_experience_years",
    ]
].head(30)

Likely entry-level postings: 11


,title,company,location,title_seniority,minimum_experience_years
12,Junior IT Operations/Reporting Analyst/Adminis...,BC Forward,"Pennington, Mercer County",Junior or Entry,<NA>
33,Data Analyst,General Dynamics Information Technology,"Westlake, Montgomery County",Unspecified,2
46,Operations Analyst,Sharp Decisions,"New York City, New York",Unspecified,0
55,DATA ANALYST,AaraTechnologies Inc,"Chesapeake City, Cecil County",Unspecified,2
107,Junior Business & Data Analyst,Ellianse LLC,"Five Corners, Hudson County",Junior or Entry,<NA>
166,Business Intelligence Analyst - Associate with...,Plateau Software Inc,"Norfolk, Norfolk City",Junior or Entry,<NA>
174,Junior Business Intelligence Analyst,Lansing Building Products,"Richmond, Richmond County",Junior or Entry,<NA>
315,Business Intelligence Analyst - Associate,Plateau Software,"Virginia Beach, Virginia Beach City",Junior or Entry,<NA>
347,Data Driven Marketing Analyst - Audience Design,Deluxe,"Grand Central, Manhattan",Unspecified,1
370,Entry-Level Brokerage Operations Analyst,Kellton,"Grand Central, Manhattan",Junior or Entry,<NA>


In [33]:
def classify_entry_accessibility(row):
    title_level = row["title_seniority"]
    years = row["minimum_experience_years"]

    if title_level == "Internship":
        return "Internship"

    if title_level == "Junior or Entry":
        return "Explicit Entry-Level"

    if title_level in [
        "Director or Executive",
        "Manager",
        "Senior or Lead",
    ]:
        return "Advanced-Level"

    if pd.isna(years):
        return "Potential Entry-Level / Unclear"

    if years <= 2:
        return "Experience-Compatible"

    if years <= 4:
        return "Mid-Level"

    return "Advanced-Level"


jobs_clean["entry_accessibility"] = jobs_clean.apply(
    classify_entry_accessibility,
    axis=1,
)

In [34]:
accessibility_counts = (
    jobs_clean["entry_accessibility"]
    .value_counts()
    .rename_axis("entry_accessibility")
    .reset_index(name="job_count")
)

accessibility_counts["percent"] = (
    accessibility_counts["job_count"]
    / len(jobs_clean)
    * 100
).round(1)

accessibility_counts

,entry_accessibility,job_count,percent
0,Potential Entry-Level / Unclear,308,70.5
1,Advanced-Level,109,24.9
2,Internship,8,1.8
3,Explicit Entry-Level,7,1.6
4,Experience-Compatible,4,0.9
5,Mid-Level,1,0.2


In [35]:
early_career_categories = [
    "Explicit Entry-Level",
    "Experience-Compatible",
    "Potential Entry-Level / Unclear",
    "Internship",
]

early_career_candidates = jobs_clean.loc[
    jobs_clean["entry_accessibility"].isin(
        early_career_categories
    )
].copy()

print(
    "Early-career candidate postings:",
    len(early_career_candidates),
)

Early-career candidate postings: 327


In [36]:
for category in early_career_categories:
    print("=" * 60)
    print(category)
    print("=" * 60)

    examples = early_career_candidates.loc[
        early_career_candidates["entry_accessibility"] == category,
        [
            "title",
            "company",
            "location",
            "minimum_experience_years",
        ],
    ].head(15)

    display(examples)

Explicit Entry-Level


,title,company,location,minimum_experience_years
12,Junior IT Operations/Reporting Analyst/Adminis...,BC Forward,"Pennington, Mercer County",<NA>
107,Junior Business & Data Analyst,Ellianse LLC,"Five Corners, Hudson County",<NA>
166,Business Intelligence Analyst - Associate with...,Plateau Software Inc,"Norfolk, Norfolk City",<NA>
174,Junior Business Intelligence Analyst,Lansing Building Products,"Richmond, Richmond County",<NA>
315,Business Intelligence Analyst - Associate,Plateau Software,"Virginia Beach, Virginia Beach City",<NA>
370,Entry-Level Brokerage Operations Analyst,Kellton,"Grand Central, Manhattan",<NA>
371,Marketing Compliance Analyst/Associate,Apollo Management Holdings,"New York City, New York",<NA>


Experience-Compatible


,title,company,location,minimum_experience_years
33,Data Analyst,General Dynamics Information Technology,"Westlake, Montgomery County",2
46,Operations Analyst,Sharp Decisions,"New York City, New York",0
55,DATA ANALYST,AaraTechnologies Inc,"Chesapeake City, Cecil County",2
347,Data Driven Marketing Analyst - Audience Design,Deluxe,"Grand Central, Manhattan",1


Potential Entry-Level / Unclear


,title,company,location,minimum_experience_years
0,Data Analyst,Brooksource,"New Jersey, US",<NA>
1,Healthcare Reporting Analyst,"Charter Global, Inc.","East Case, Baltimore",<NA>
2,Reporting Analyst,System One,"Druid, Baltimore",<NA>
3,Big Data Analyst,Vantor,"Herndon, Fairfax County",<NA>
5,Reporting Analyst,Vega Consulting Solutions,"East Case, Baltimore",<NA>
6,Program Operations Analyst with Security Clear...,Quantum Sky,"Suitland, Prince George's County",<NA>
7,Data Analyst,International Solutions Group,"Pimmit, Fairfax County",<NA>
8,Mainframe Data Analyst,TECH Tammina,"Flatbush, Brooklyn",<NA>
9,Reporting Analyst,Lumen Solutions Group Inc.,"East Case, Baltimore",<NA>
10,Business Intelligence Analyst,Asset Based Lending LLC,"Cherry Hill, Camden County",<NA>


Internship


,title,company,location,minimum_experience_years
138,M&D Operations Analyst Intern - OVIP,Oracle,"Adelphi, Prince George's County",<NA>
139,M&D Operations Analyst Intern - OVIP,Oracle,"Elizabeth, Union County",<NA>
140,M&D Operations Analyst Intern - OVIP,Oracle,"Trenton, Mercer County",<NA>
141,M&D Operations Analyst Intern - OVIP,Oracle,"New York City, New York",<NA>
142,M&D Operations Analyst Intern - OVIP,Oracle,"Bronx, New York City",<NA>
143,M&D Operations Analyst Intern - OVIP,Oracle,"Newark, Essex County",<NA>
144,M&D Operations Analyst Intern - OVIP,Oracle,"Jersey City, Hudson County",<NA>
145,M&D Operations Analyst Intern - OVIP,Oracle,"Paterson, Passaic County",<NA>


In [38]:
def classify_role_relevance(title):
    """Identify whether a title belongs to the target analytics roles."""
    title_text = normalize_text(title)

    target_patterns = [
        r"\bdata analyst\b",
        r"\bbusiness intelligence analyst\b",
        r"\bbi analyst\b",
        r"\breporting analyst\b",
        r"\boperations analyst\b",
        r"\bmarketing analyst\b",
        r"\bdigital marketing analyst\b",
        r"\banalytics analyst\b",
        r"\binsights analyst\b",
        r"\bresearch analyst\b",
        r"\bbusiness analyst\b",
    ]

    exclusion_patterns = [
        r"\bradiologic technologist\b",
        r"\bsoftware engineer\b",
        r"\bfullstack\b",
        r"\bfull stack\b",
        r"\benterprise architect\b",
        r"\bdata architect\b",
        r"\bproduct manager\b",
        r"\bsales development manager\b",
        r"\baccounting manager\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in exclusion_patterns
    ):
        return "Not Target Role"

    if any(
        re.search(pattern, title_text)
        for pattern in target_patterns
    ):
        return "Target Analytics Role"

    return "Needs Review"


jobs_clean["role_relevance"] = jobs_clean["title"].apply(
    classify_role_relevance
)

In [39]:
role_relevance_counts = (
    jobs_clean["role_relevance"]
    .value_counts()
    .rename_axis("role_relevance")
    .reset_index(name="job_count")
)

role_relevance_counts["percent"] = (
    role_relevance_counts["job_count"]
    / len(jobs_clean)
    * 100
).round(1)

role_relevance_counts

,role_relevance,job_count,percent
0,Target Analytics Role,311,71.2
1,Needs Review,119,27.2
2,Not Target Role,7,1.6


In [40]:
jobs_clean.loc[
    jobs_clean["role_relevance"] == "Needs Review",
    ["title", "company", "location"],
].drop_duplicates(
    subset="title"
).sort_values(
    "title"
).head(50)

,title,company,location
434,(Hiring) Marketing and Business Development An...,Viper Staffing Services,"Hackensack, Bergen County"
248,Aladdin SME/ Data Analysts,New York Technology Partners,"Grand Central, Manhattan"
123,"Analyst Hotel Level Marketing, Agency Solutions",Hilton,"Tysons Corner, Fairfax County"
421,Analyst Marketing Operations,T. Rowe Price,"Owings Mills, Baltimore County"
419,"Analyst, Contact Center Services (Trader)",T. Rowe Price,"Owings Mills, Baltimore County"
310,"Analyst, Digital Marketing",Stagwell Global LLC,"State Farm, Arlington County"
420,"Analyst, Investment Liaison - Private Asset Ma...",T. Rowe Price,"Patterson, Baltimore"
318,"Analyst, Marketing Automation",CMI Media Group,"Cherry Hill, Camden County"
427,"Analyst, Marketing Intelligence",T. Rowe Price,"Owings Mills, Baltimore County"
177,Associate Director of Digital Marketing,Worldwide Assurance for Employees of Public Ag...,"West Falls Church, Fairfax County"


In [41]:
def identify_access_restrictions(row):
    """Flag postings mentioning clearance or citizenship restrictions."""
    combined_text = normalize_text(
        f"{row['title']} {row['description']}"
    )

    clearance_pattern = (
        r"\bsecurity clearance\b|"
        r"\btop secret\b|"
        r"\bts/sci\b|"
        r"\bsecret clearance\b|"
        r"\bactive clearance\b"
    )

    citizenship_pattern = (
        r"\bu\.?s\.? citizenship required\b|"
        r"\bmust be a u\.?s\.? citizen\b|"
        r"\bus citizen only\b|"
        r"\bunited states citizen\b"
    )

    return pd.Series(
        {
            "clearance_flag": bool(
                re.search(clearance_pattern, combined_text)
            ),
            "citizenship_flag": bool(
                re.search(citizenship_pattern, combined_text)
            ),
        }
    )


restriction_flags = jobs_clean.apply(
    identify_access_restrictions,
    axis=1,
)

jobs_clean = pd.concat(
    [jobs_clean, restriction_flags],
    axis=1,
)

jobs_clean["restricted_access_flag"] = (
    jobs_clean["clearance_flag"]
    | jobs_clean["citizenship_flag"]
)

In [42]:
restriction_by_state = (
    jobs_clean.groupby("matched_query_states")
    .agg(
        total_jobs=("job_id", "count"),
        restricted_jobs=("restricted_access_flag", "sum"),
    )
    .reset_index()
)

restriction_by_state["restricted_percent"] = (
    restriction_by_state["restricted_jobs"]
    / restriction_by_state["total_jobs"]
    * 100
).round(1)

restriction_by_state.sort_values(
    "restricted_percent",
    ascending=False,
)

,matched_query_states,total_jobs,restricted_jobs,restricted_percent
0,DC,70,19,27.1
2,DC | VA,18,4,22.2
7,VA,79,16,20.3
3,MD,85,17,20.0
1,DC | MD,7,1,14.3
4,NJ,82,0,0.0
5,NJ | NY,11,0,0.0
6,NY,85,0,0.0


In [43]:
confirmed_early_career = jobs_clean.loc[
    jobs_clean["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Internship",
        ]
    )
    & (
        jobs_clean["role_relevance"]
        == "Target Analytics Role"
    )
].copy()

potential_early_career = jobs_clean.loc[
    jobs_clean["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Potential Entry-Level / Unclear",
            "Internship",
        ]
    )
    & (
        jobs_clean["role_relevance"]
        == "Target Analytics Role"
    )
].copy()

print(
    "Confirmed early-career analytics postings:",
    len(confirmed_early_career),
)

print(
    "Potential early-career analytics postings:",
    len(potential_early_career),
)

print(
    "Confirmed postings without access restrictions:",
    (~confirmed_early_career["restricted_access_flag"]).sum(),
)

Confirmed early-career analytics postings: 18
Potential early-career analytics postings: 246
Confirmed postings without access restrictions: 16


In [44]:
needs_review_titles = (
    jobs_clean.loc[
        jobs_clean["role_relevance"] == "Needs Review",
        ["title", "company", "location"],
    ]
    .drop_duplicates(subset=["title"])
    .sort_values("title")
    .reset_index(drop=True)
)

print("Unique titles needing review:", len(needs_review_titles))

pd.set_option("display.max_rows", 150)
needs_review_titles

Unique titles needing review: 70


,title,company,location
0,(Hiring) Marketing and Business Development An...,Viper Staffing Services,"Hackensack, Bergen County"
1,Aladdin SME/ Data Analysts,New York Technology Partners,"Grand Central, Manhattan"
2,"Analyst Hotel Level Marketing, Agency Solutions",Hilton,"Tysons Corner, Fairfax County"
3,Analyst Marketing Operations,T. Rowe Price,"Owings Mills, Baltimore County"
4,"Analyst, Contact Center Services (Trader)",T. Rowe Price,"Owings Mills, Baltimore County"
5,"Analyst, Digital Marketing",Stagwell Global LLC,"State Farm, Arlington County"
6,"Analyst, Investment Liaison - Private Asset Ma...",T. Rowe Price,"Patterson, Baltimore"
7,"Analyst, Marketing Automation",CMI Media Group,"Cherry Hill, Camden County"
8,"Analyst, Marketing Intelligence",T. Rowe Price,"Owings Mills, Baltimore County"
9,Associate Director of Digital Marketing,Worldwide Assurance for Employees of Public Ag...,"West Falls Church, Fairfax County"


In [45]:
needs_review_frequency = (
    jobs_clean.loc[
        jobs_clean["role_relevance"] == "Needs Review",
        "title",
    ]
    .value_counts()
    .rename_axis("title")
    .reset_index(name="job_count")
)

needs_review_frequency.head(50)

,title,job_count
0,Operations System Analyst,26
1,Financial Services - Customer Tax Operations a...,6
2,Data Platform Evangelist,6
3,Senior Paid Media Analyst [Remote],3
4,Private Markets Marketing/Fundraising Analyst,2
5,Principal Financial Analyst - SEC Financial Re...,2
6,Data Scientist,2
7,"Senior Analyst, Business Intelligence",2
8,Chief Enterprise Business Architect (Business ...,2
9,External Reporting Controller - Analyst,2


In [49]:
def classify_role_category(title):
    """Classify a job title into a broad analytics category."""
    title_text = normalize_text(title)

    # Clearly unrelated to the project's target market
    non_target_patterns = [
        r"\blicensed master social worker\b",
        r"\breimbursement specialist\b",
        r"\bdod skillbridge\b",
        r"\btarget analyst reporter\b",
        r"\btarget digital network reporter\b",
        r"\bcomputer security incident report analyst\b",
        r"\bcontact center services.*trader\b",
        r"\bsales development manager\b",
        r"\bbusiness strategist.*government intelligence\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in non_target_patterns
    ):
        return "Not Target Role"

    # Marketing, advertising, CRM, customer, and audience analytics
    marketing_patterns = [
        r"\bmarketing\b",
        r"\badvertising research\b",
        r"\bmarket research\b",
        r"\bmedia effectiveness\b",
        r"\bpaid media\b",
        r"\bmarketing mix\b",
        r"\bcustomer experience insights\b",
        r"\bcustomer insights\b",
        r"\bcategory insights\b",
        r"\baudience\b",
        r"\bcrm analyst\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in marketing_patterns
    ):
        return "Marketing and Insights"

    # Finance, investment, compliance, tax, and risk analytics
    financial_patterns = [
        r"\bfinancial\b",
        r"\binvestment\b",
        r"\bequity analyst\b",
        r"\bcredit operations\b",
        r"\bfinancial crimes\b",
        r"\brisk analyst\b",
        r"\bpricing analyst\b",
        r"\btax operations\b",
        r"\bprivate markets\b",
        r"\bliquidity reporting\b",
        r"\bregulatory reporting\b",
        r"\bcontroller.*analyst\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in financial_patterns
    ):
        return "Financial and Risk Analytics"

    # Data science roles
    if re.search(r"\bdata scientist\b", title_text):
        return "Data Science"

    # Core data analyst roles
    data_analytics_patterns = [
        r"\bdata analysts?\b",
        r"\bdata management analyst\b",
        r"\bdata quality analyst\b",
        r"\bdata governance analyst\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in data_analytics_patterns
    ):
        return "Data Analytics"

    # BI, reporting, and dashboard-related roles
    bi_reporting_patterns = [
        r"\bbusiness intelligence\b",
        r"\bbi analyst\b",
        r"\breporting analyst\b",
        r"\breporting and analytics\b",
        r"\banalytics analyst\b",
        r"\bstore reporting\b",
        r"\bservice reporting\b",
        r"\breporting controller\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in bi_reporting_patterns
    ):
        return "BI and Reporting"

    # Business, process, systems, program, and operations analysts
    business_operations_patterns = [
        r"\boperations analyst\b",
        r"\boperations system analyst\b",
        r"\bbusiness process analyst\b",
        r"\bbusiness systems analyst\b",
        r"\bbusiness analyst\b",
        r"\bprogram analyst\b",
        r"\bproduct analyst\b",
        r"\bstrategic options\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in business_operations_patterns
    ):
        return "Business and Operations"

    # Technical data jobs that are related but outside the main analyst scope
    adjacent_technical_patterns = [
        r"\bdata modeler\b",
        r"\bdatabase administrator\b",
        r"\bdata engineer\b",
        r"\bbig data engineer\b",
        r"\bdata architect\b",
        r"\benterprise architect\b",
        r"\bdata platform evangelist\b",
        r"\bdigital platforms.*technology.*analyst\b",
        r"\bprogrammer analyst\b",
    ]

    if any(
        re.search(pattern, title_text)
        for pattern in adjacent_technical_patterns
    ):
        return "Adjacent Technical Data Role"

    return "Needs Review"


jobs_clean["role_category"] = jobs_clean["title"].apply(
    classify_role_category
)

In [50]:
role_category_counts = (
    jobs_clean["role_category"]
    .value_counts()
    .rename_axis("role_category")
    .reset_index(name="job_count")
)

role_category_counts["percent"] = (
    role_category_counts["job_count"]
    / len(jobs_clean)
    * 100
).round(1)

role_category_counts

,role_category,job_count,percent
0,BI and Reporting,102,23.3
1,Data Analytics,97,22.2
2,Business and Operations,90,20.6
3,Marketing and Insights,82,18.8
4,Financial and Risk Analytics,30,6.9
5,Not Target Role,16,3.7
6,Adjacent Technical Data Role,10,2.3
7,Needs Review,6,1.4
8,Data Science,4,0.9


In [52]:
jobs_clean.loc[
    jobs_clean["role_category"] == "Needs Review",
    ["title", "company", "location"],
].drop_duplicates(
    subset="title"
).sort_values(
    "title"
).head(100)

,title,company,location
336,Mid-Level Fullstack Software Engineer,Software Guidance & Assistance,"Five Corners, Hudson County"
14,Radiologic Technologist,Atlas Search,"New York City, New York"
57,Registered Nurse RN,Atlas Search,"New York City, New York"
79,"Senior Manager, Business Development Analytics",Bristol Myers Squibb,"Princeton, Mercer County"
95,"Supervisor, Reporting & Analytics",Ttec,"Washington, D.C., US"


In [53]:
target_role_categories = [
    "Data Analytics",
    "BI and Reporting",
    "Marketing and Insights",
    "Business and Operations",
    "Financial and Risk Analytics",
    "Data Science",
]


def convert_category_to_relevance(category):
    if category in target_role_categories:
        return "Target Analytics Role"

    if category == "Adjacent Technical Data Role":
        return "Adjacent Technical Role"

    if category == "Not Target Role":
        return "Not Target Role"

    return "Needs Review"


jobs_clean["role_relevance"] = jobs_clean[
    "role_category"
].apply(convert_category_to_relevance)

In [54]:
role_category_counts = (
    jobs_clean["role_category"]
    .value_counts()
    .rename_axis("role_category")
    .reset_index(name="job_count")
)

role_category_counts["percent"] = (
    role_category_counts["job_count"]
    / len(jobs_clean)
    * 100
).round(1)

role_category_counts

,role_category,job_count,percent
0,BI and Reporting,102,23.3
1,Data Analytics,97,22.2
2,Business and Operations,90,20.6
3,Marketing and Insights,82,18.8
4,Financial and Risk Analytics,30,6.9
5,Not Target Role,16,3.7
6,Adjacent Technical Data Role,10,2.3
7,Needs Review,6,1.4
8,Data Science,4,0.9


In [55]:
jobs_clean["role_relevance"].value_counts()

role_relevance
Target Analytics Role      405
Not Target Role             16
Adjacent Technical Role     10
Needs Review                 6
Name: count, dtype: int64

In [56]:
def classify_title_seniority(title):
    """Classify seniority using keywords in the job title."""
    title_text = normalize_text(title)

    executive_pattern = (
        r"\b(director|vice president|vp|avp|chief|head)\b"
    )

    manager_pattern = (
        r"\b(manager|supervisor)\b"
    )

    senior_pattern = (
        r"\b(senior|sr\.?|lead|principal)\b|"
        r"\bsubject matter expert\b|"
        r"\bsme\b"
    )

    mid_level_pattern = (
        r"\bmid[- ]level\b|"
        r"\bmid\b"
    )

    internship_pattern = (
        r"\b(intern|internship|co-op)\b"
    )

    entry_pattern = (
        r"\b(junior|jr\.?|entry[- ]level|"
        r"associate|trainee|graduate)\b|"
        r"\blevel\s*(?:1|i)\b|"
        r"\banalyst\s*(?:1|i)\b"
    )

    staff_pattern = r"\bstaff\b"

    if re.search(executive_pattern, title_text):
        return "Director or Executive"

    if re.search(manager_pattern, title_text):
        return "Manager"

    if re.search(senior_pattern, title_text):
        return "Senior or Lead"

    if re.search(mid_level_pattern, title_text):
        return "Mid-Level"

    if re.search(internship_pattern, title_text):
        return "Internship"

    if re.search(entry_pattern, title_text):
        return "Junior or Entry"

    if re.search(staff_pattern, title_text):
        return "Staff or Unclear"

    return "Unspecified"


jobs_clean["title_seniority"] = jobs_clean["title"].apply(
    classify_title_seniority
)

In [57]:
def classify_entry_accessibility(row):
    title_level = row["title_seniority"]
    years = row["minimum_experience_years"]

    if title_level == "Internship":
        return "Internship"

    if title_level == "Junior or Entry":
        return "Explicit Entry-Level"

    if title_level == "Mid-Level":
        return "Mid-Level"

    if title_level in [
        "Director or Executive",
        "Manager",
        "Senior or Lead",
    ]:
        return "Advanced-Level"

    if pd.isna(years):
        return "Potential Entry-Level / Unclear"

    if years <= 2:
        return "Experience-Compatible"

    if years <= 4:
        return "Mid-Level"

    return "Advanced-Level"


jobs_clean["entry_accessibility"] = jobs_clean.apply(
    classify_entry_accessibility,
    axis=1,
)

In [58]:
remaining_review = (
    jobs_clean.loc[
        jobs_clean["role_category"] == "Needs Review",
        ["title", "company", "location"],
    ]
    .drop_duplicates(subset="title")
    .sort_values("title")
    .reset_index(drop=True)
)

print("Remaining unique titles needing review:", len(remaining_review))

remaining_review

Remaining unique titles needing review: 5


,title,company,location
0,Mid-Level Fullstack Software Engineer,Software Guidance & Assistance,"Five Corners, Hudson County"
1,Radiologic Technologist,Atlas Search,"New York City, New York"
2,Registered Nurse RN,Atlas Search,"New York City, New York"
3,"Senior Manager, Business Development Analytics",Bristol Myers Squibb,"Princeton, Mercer County"
4,"Supervisor, Reporting & Analytics",Ttec,"Washington, D.C., US"


In [61]:
manual_role_overrides = {
    "Mid-Level Fullstack Software Engineer": "Not Target Role",
    "Radiologic Technologist": "Not Target Role",
    "Registered Nurse RN": "Not Target Role",
    "Senior Manager, Business Development Analytics": (
        "Business and Operations"
    ),
    "Supervisor, Reporting & Analytics": "BI and Reporting",
}

jobs_clean["role_category"] = jobs_clean.apply(
    lambda row: manual_role_overrides.get(
        row["title"],
        row["role_category"],
    ),
    axis=1,
)

In [62]:
jobs_clean["role_relevance"] = jobs_clean[
    "role_category"
].apply(convert_category_to_relevance)

In [63]:
print(
    "Titles still needing review:",
    (jobs_clean["role_category"] == "Needs Review").sum(),
)

jobs_clean["role_category"].value_counts()

Titles still needing review: 0


role_category
BI and Reporting                103
Data Analytics                   97
Business and Operations          92
Marketing and Insights           82
Financial and Risk Analytics     30
Not Target Role                  19
Adjacent Technical Data Role     10
Data Science                      4
Name: count, dtype: int64

In [64]:
confirmed_early_career = jobs_clean.loc[
    jobs_clean["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Internship",
        ]
    )
    & (
        jobs_clean["role_relevance"]
        == "Target Analytics Role"
    )
].copy()

potential_early_career = jobs_clean.loc[
    jobs_clean["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Potential Entry-Level / Unclear",
            "Internship",
        ]
    )
    & (
        jobs_clean["role_relevance"]
        == "Target Analytics Role"
    )
].copy()

print(
    "Confirmed early-career analytics postings:",
    len(confirmed_early_career),
)

print(
    "Potential early-career analytics postings:",
    len(potential_early_career),
)

print(
    "Confirmed postings without access restrictions:",
    (~confirmed_early_career["restricted_access_flag"]).sum(),
)

Confirmed early-career analytics postings: 22
Potential early-career analytics postings: 297
Confirmed postings without access restrictions: 19


In [65]:
analysis_jobs = jobs_clean.copy()

# Convert date columns to datetime
analysis_jobs["created"] = pd.to_datetime(
    analysis_jobs["created"],
    errors="coerce",
    utc=True,
)

analysis_jobs["collected_at_utc"] = pd.to_datetime(
    analysis_jobs["collected_at_utc"],
    errors="coerce",
    utc=True,
)

# Final analysis flags
analysis_jobs["target_role_flag"] = (
    analysis_jobs["role_relevance"]
    == "Target Analytics Role"
)

analysis_jobs["confirmed_early_career_flag"] = (
    analysis_jobs["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Internship",
        ]
    )
    & analysis_jobs["target_role_flag"]
)

analysis_jobs["potential_early_career_flag"] = (
    analysis_jobs["entry_accessibility"].isin(
        [
            "Explicit Entry-Level",
            "Experience-Compatible",
            "Potential Entry-Level / Unclear",
            "Internship",
        ]
    )
    & analysis_jobs["target_role_flag"]
)

analysis_jobs["accessible_confirmed_early_career_flag"] = (
    analysis_jobs["confirmed_early_career_flag"]
    & ~analysis_jobs["restricted_access_flag"]
)

In [66]:
helper_columns = [
    "title_norm",
    "company_norm",
    "location_norm",
    "description_norm",
    "salary_min_key",
    "salary_max_key",
    "created_key",
    "strict_duplicate_key",
]

analysis_jobs = analysis_jobs.drop(
    columns=helper_columns,
    errors="ignore",
)

analysis_jobs = analysis_jobs.sort_values(
    [
        "role_category",
        "matched_query_states",
        "title",
    ]
).reset_index(drop=True)

In [67]:
print("Final dataset shape:", analysis_jobs.shape)
print("Unique job IDs:", analysis_jobs["job_id"].nunique())
print(
    "Target analytics roles:",
    analysis_jobs["target_role_flag"].sum(),
)
print(
    "Confirmed early-career roles:",
    analysis_jobs["confirmed_early_career_flag"].sum(),
)
print(
    "Potential early-career roles:",
    analysis_jobs["potential_early_career_flag"].sum(),
)
print(
    "Accessible confirmed early-career roles:",
    analysis_jobs[
        "accessible_confirmed_early_career_flag"
    ].sum(),
)

Final dataset shape: (437, 36)
Unique job IDs: 437
Target analytics roles: 408
Confirmed early-career roles: 22
Potential early-career roles: 297
Accessible confirmed early-career roles: 19


In [70]:
category_audit = (
    analysis_jobs["role_category"]
    .value_counts()
    .rename_axis("role_category")
    .reset_index(name="job_count")
)

category_audit["percent"] = (
    category_audit["job_count"]
    / len(analysis_jobs)
    * 100
).round(1)

category_audit

,role_category,job_count,percent
0,BI and Reporting,103,23.6
1,Data Analytics,97,22.2
2,Business and Operations,92,21.1
3,Marketing and Insights,82,18.8
4,Financial and Risk Analytics,30,6.9
5,Not Target Role,19,4.3
6,Adjacent Technical Data Role,10,2.3
7,Data Science,4,0.9


In [71]:
analysis_jobs.loc[
    ~analysis_jobs["target_role_flag"],
    [
        "title",
        "company",
        "role_category",
        "role_relevance",
    ],
].sort_values(
    ["role_category", "title"]
)

,title,company,role_category,role_relevance
2,Data Modeler,Cymertek,Adjacent Technical Data Role,Adjacent Technical Role
0,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
1,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
3,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
4,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
8,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
9,Data Platform Evangelist,Oracle,Adjacent Technical Data Role,Adjacent Technical Role
5,Database Administrator,Cymertek,Adjacent Technical Data Role,Adjacent Technical Role
7,Digital Platforms & Technology Senior Analyst,DTCC,Adjacent Technical Data Role,Adjacent Technical Role
6,Java AWS Big Data Engineer,Software Guidance & Assistance,Adjacent Technical Data Role,Adjacent Technical Role


In [72]:
for category in target_role_categories:
    print("=" * 70)
    print(category)
    print("=" * 70)

    examples = analysis_jobs.loc[
        analysis_jobs["role_category"] == category,
        [
            "title",
            "company",
            "location",
            "title_seniority",
        ],
    ].sample(
        n=min(
            10,
            (
                analysis_jobs["role_category"]
                == category
            ).sum(),
        ),
        random_state=42,
    )

    display(examples)

Data Analytics


,title,company,location,title_seniority
267,Business Data Analyst,AIG Insurance,"Jersey City, Hudson County",Unspecified
245,"Submarine Logistics Data Analyst, Lead","BOOZ, ALLEN & HAMILTON, INC.","Bowie, Prince George's County",Senior or Lead
298,IT Data Analyst,Apex Systems,"Fairfax, Fairfax County",Unspecified
223,Data Analyst,Cherokee Federal,"State Farm, Arlington County",Unspecified
286,Senior Data Analyst with ETL expertise,UNIVERSAL Technologies,"Flatbush, Brooklyn",Senior or Lead
288,Big Data Analyst,Vantor,"Herndon, Fairfax County",Unspecified
269,Junior Business & Data Analyst,Ellianse LLC,"Five Corners, Hudson County",Junior or Entry
247,Business Data Analyst,Fiserv,"Berkeley Heights, Union County",Unspecified
215,"Submarine Logistics Data Analyst, Lead","BOOZ, ALLEN & HAMILTON, INC.","Washington, Washington, D.C.",Senior or Lead
205,Data Analyst,CACI International,"Washington, D.C., US",Unspecified


BI and Reporting


,title,company,location,title_seniority
40,Reporting Analyst,System One,"Druid, Baltimore",Unspecified
77,"Senior Analyst, Business Intelligence",Horizon Media,"New York City, New York",Senior or Lead
72,Data Quality & Reporting Analyst (Commercial R...,The Community Preservation Corporation,"New York City, New York",Unspecified
57,REMOTE Default Reporting and Analytics Analyst,Carrington,"Ellisburg, Camden County",Unspecified
52,Cyber Risk Governance & Reporting Analyst,Fiserv,"Berkeley Heights, Union County",Unspecified
50,Business Intelligence Analyst,Sole Solutions,"Paramus, Bergen County",Unspecified
100,Market & Business Intelligence Analyst,DLA Piper,"Reston, Fairfax County",Unspecified
55,Government Reporting Analyst,SourcePro Search,"Wayne, Passaic County",Unspecified
20,Senior Business Intelligence Analyst- Tableau ...,SitusAMC,"Washington, D.C., US",Senior or Lead
10,BI Analyst II,Indeed,"Washington, D.C., US",Unspecified


Marketing and Insights


...

,title,company,location,title_seniority
153,"Associate Director, Strategic Options and Asse...",Bristol Myers Squibb,"Princeton, Mercer County",Director or Executive
135,Submarine Military Operations Analyst with Sec...,SPA,"Alexandria, Alexandria City",Unspecified
168,"Senior Manager, Business Development Analytics",Bristol Myers Squibb,"Princeton, Mercer County",Manager
185,Sales Operations Analyst,Wolters Kluwer,"Grand Central, Manhattan",Unspecified
113,(697) HR Operations Analyst with Security Clea...,Arlo Solutions,"Washington, D.C., US",Unspecified
139,Operations Analyst,"Campbell & Company, LP","Towson, Baltimore County",Unspecified
152,"Associate Director, Strategic Options and Asse...",Bristol Myers Squibb,"Princeton, Mercer County",Director or Executive
180,M&D Operations Analyst Intern - OVIP,Oracle,"Bronx, New York City",Internship
123,Operations Analyst III with Security Clearance,enGenius Consulting Group Inc,"South, Arlington County",Unspecified
157,Deposit Operations Analyst,Gpac,"Lawrence Township, Mercer County",Unspecified


Financial and Risk Analytics


,title,company,location,title_seniority
333,Financial Reporting Analyst,Aston Carter,"Virginia Beach, Virginia Beach City",Unspecified
321,Financial Services - Customer Tax Operations a...,EY,"Iselin, Middlesex County",Senior or Lead
329,Regulatory Reporting Business Analyst,TheStaffed,"New York City, New York",Unspecified
323,ISO Regulatory Reporting Analyst,EXL,"New Jersey, US",Unspecified
314,DoW Financial Reporting Data Analyst | 5 yrs |...,"Silverthorne Advisory Group, LLC","Trade, Alexandria City",Unspecified
315,"Analyst, Investment Liaison - Private Asset Ma...",T. Rowe Price,"Patterson, Baltimore",Unspecified
334,Financial Reporting Analyst Senior - Departmen...,VCU Health,"Richmond, Richmond County",Senior or Lead
330,Tax Operations Analyst,SMBC,"Grand Central, Manhattan",Unspecified
318,Financial Reporting Analyst III,Avis Budget Group,"Parsippany, Morris County",Unspecified
306,Credit Operations & Reporting Senior Analyst,Banc of California,"Martins Additions, Montgomery County",Senior or Lead


Data Science


,title,company,location,title_seniority
303,Data Scientist - Mid,Cherokee Federal,"Mount Rainier, Prince George's County",Mid-Level
305,Data Scientist,Cymertek,"Annapolis Junction, Anne Arundel County",Unspecified
302,Data Scientist - Mid with Security Clearance,Cherokee Federal,"Washington, D.C., US",Mid-Level
304,Data Scientist,Cymertek,"Annapolis Junction, Anne Arundel County",Unspecified


In [73]:
collection_stamp = latest_file.stem.replace(
    "adzuna_jobs_",
    "",
)

processed_output_path = (
    PROCESSED_DATA_DIR
    / f"adzuna_jobs_cleaned_{collection_stamp}.csv"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

analysis_jobs.to_csv(
    processed_output_path,
    index=False,
)

print("Saved cleaned dataset to:")
print(processed_output_path)

Saved cleaned dataset to:
/Users/imseoyeong/Desktop/east-coast-job-market-analysis/data/processed/adzuna_jobs_cleaned_20260804_2023.csv


In [74]:
role_summary = (
    analysis_jobs.groupby("role_category")
    .agg(
        total_jobs=("job_id", "count"),
        confirmed_early_career=(
            "confirmed_early_career_flag",
            "sum",
        ),
        potential_early_career=(
            "potential_early_career_flag",
            "sum",
        ),
        restricted_jobs=(
            "restricted_access_flag",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "total_jobs",
        ascending=False,
    )
)

role_summary["percent_of_all_jobs"] = (
    role_summary["total_jobs"]
    / len(analysis_jobs)
    * 100
).round(1)

role_summary

,role_category,total_jobs,confirmed_early_career,potential_early_career,restricted_jobs,percent_of_all_jobs
1,BI and Reporting,103,5,72,10,23.6
3,Data Analytics,97,4,74,16,22.2
2,Business and Operations,92,11,79,14,21.1
6,Marketing and Insights,82,2,58,4,18.8
5,Financial and Risk Analytics,30,0,12,2,6.9
7,Not Target Role,19,0,0,5,4.3
0,Adjacent Technical Data Role,10,0,0,2,2.3
4,Data Science,4,0,2,4,0.9
